**Celda 1 configuracion**

In [16]:
# ============================================================
# CELDA 1 - CONFIGURACION
# ============================================================

# Si ya instalaste semantic-link-labs, NO vuelvas a ejecutar %pip.
# Si es un notebook nuevo y falla el import, ejecuta en una celda aparte:
# %pip install semantic-link-labs

import re
import copy
import pandas as pd
from IPython.display import display

import sempy_labs as labs
import sempy_labs.dataflow as dataflow
from sempy_labs._helper_functions import resolve_workspace_name_and_id

# ------------------------------------------------------------
# 1. MODELO SEMANTICO A MIGRAR
# ------------------------------------------------------------

SEMANTIC_WORKSPACE = "Challenge_SemanticLink_Labs"
SEMANTIC_MODEL = "ChallengeSemnaticLink"

# ------------------------------------------------------------
# 2. MAPEO DATAFLOW GEN1 -> DATAFLOW GEN2
# ------------------------------------------------------------
# Pon aqui todos los dataflows que quieras migrar.
# Izquierda: nombre del Dataflow Gen1.
# Derecha: nombre del Dataflow Gen2.
#
# Si dejas un Gen1 sin mapear, el codigo intentara buscar automaticamente:
# NombreGen1 + "_1"

GEN2_NAME_MAPPING = {
    "DatosCliente": "DatosCliente_1"
}

# ------------------------------------------------------------
# 3. CONTROLES DE SEGURIDAD
# ------------------------------------------------------------

# Primera ejecucion recomendada:
# APPLY_CHANGES = False
# APPLY_REBIND = False
# RUN_REFRESH = False

# Cuando veas bien 02_mapeo_gen1_a_gen2 y 03_cambios_propuestos:
# APPLY_CHANGES = True

# Cuando ya se actualizo el BIM y quieres conectar/refrescar:
# APPLY_REBIND = True
# RUN_REFRESH = True

APPLY_CHANGES = True
CREATE_BACKUP_MODEL = True
BACKUP_MODEL_NAME = f"{SEMANTIC_MODEL}_BACKUP_ANTES_DATAFLOW_GEN2"

APPLY_REBIND = True
RUN_REFRESH = True
TAKE_OVER_MODEL = False

# ------------------------------------------------------------
# 4. SELECCION DE CONEXION FABRIC PARA REBIND
# ------------------------------------------------------------
# Si sabes exactamente que conexion quieres usar, pon su Connection Id.
# Si lo dejas None, el codigo intenta elegir la mejor automaticamente.

PREFERRED_CONNECTION_ID = None

# Opcional: filtrar por parte del nombre de la conexion.
# Ejemplo: "DatosCliente", "Dataflow", "PowerPlatform"
PREFERRED_CONNECTION_NAME_CONTAINS = None

PREFERRED_CONNECTION_TYPE_KEYWORDS = [
    "PowerPlatformDataflows",
    "PowerBIDataflows",
    "Dataflows",
    "Dataflow",
]

# ------------------------------------------------------------
# 5. FUNCIONES COMUNES
# ------------------------------------------------------------

def show(title, value):
    print(f"\n==================== {title} ====================")
    if isinstance(value, pd.DataFrame):
        print(f"{len(value):,} filas")
        display(value)
    else:
        print(value)
    return value

def s(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value)

def has_any(value, keywords):
    value = s(value).lower()
    return any(k.lower() in value for k in keywords)

def get_first_existing(row, cols):
    for col in cols:
        if col in row.index and s(row[col]):
            return s(row[col])
    return ""

def expression_to_text(expr):
    if isinstance(expr, list):
        return "\n".join(expr)
    if isinstance(expr, str):
        return expr
    return ""

def text_to_expression(original_expr, text):
    if isinstance(original_expr, list):
        return text.split("\n")
    return text

print("Configuracion cargada.")
print("Workspace:", SEMANTIC_WORKSPACE)
print("Modelo semantico:", SEMANTIC_MODEL)
print("APPLY_CHANGES:", APPLY_CHANGES)
print("APPLY_REBIND:", APPLY_REBIND)
print("RUN_REFRESH:", RUN_REFRESH)


StatementMeta(, 908309ba-3c8b-4769-a868-a8529552e783, 47, Finished, Available, Finished, False)

Configuracion cargada.
Workspace: Challenge_SemanticLink_Labs
Modelo semantico: ChallengeSemnaticLink
APPLY_CHANGES: True
APPLY_REBIND: True
RUN_REFRESH: True


**Celda 2 Migracion**

In [15]:
# ============================================================
# CELDA 2 - MIGRAR DATAFLOW GEN1 -> DATAFLOW GEN2 EN EL BIM
# ============================================================

def extract_dataflow_ids_from_m(text):
    workspace_ids = re.findall(r'workspaceId\s*=\s*"([^"]+)"', text, flags=re.I)
    dataflow_ids = re.findall(r'dataflowId\s*=\s*"([^"]+)"', text, flags=re.I)
    entities = re.findall(r'entity\s*=\s*"([^"]+)"', text, flags=re.I)

    connector = None
    if "PowerBI.Dataflows" in text:
        connector = "PowerBI.Dataflows"
    elif "PowerPlatform.Dataflows" in text:
        connector = "PowerPlatform.Dataflows"

    rows = []
    max_len = max(len(workspace_ids), len(dataflow_ids), len(entities), 1)

    for i in range(max_len):
        rows.append({
            "Connector": connector,
            "Workspace Id": workspace_ids[i] if i < len(workspace_ids) else None,
            "Dataflow Id": dataflow_ids[i] if i < len(dataflow_ids) else None,
            "Entity": entities[i] if i < len(entities) else None,
        })

    return rows

def get_all_dataflows_by_workspace(workspace_id):
    try:
        df = dataflow.list_dataflows(workspace=workspace_id)
        df["Dataflow Id Lower"] = df["Dataflow Id"].astype(str).str.lower()
        return df
    except Exception as e:
        return pd.DataFrame([{
            "Dataflow Id": None,
            "Dataflow Name": None,
            "Generation": None,
            "Error": str(e),
            "Dataflow Id Lower": None,
        }])

def find_gen2_candidate(gen1_name, all_dataflows):
    target_name = GEN2_NAME_MAPPING.get(gen1_name)

    if not target_name:
        target_name = f"{gen1_name}_1"

    candidates = all_dataflows[
        (all_dataflows["Dataflow Name"] == target_name)
        & (all_dataflows["Generation"].astype(str).str.startswith("Gen2"))
    ]

    if candidates.empty:
        candidates = all_dataflows[
            (all_dataflows["Dataflow Name"] == gen1_name)
            & (all_dataflows["Generation"].astype(str).str.startswith("Gen2"))
        ]

    if candidates.empty:
        return None

    return candidates.iloc[0].to_dict()

def audit_model_dataflow_sources(bim):
    model = bim.get("model", {})
    rows = []

    for table in model.get("tables", []):
        table_name = table.get("name")

        for partition in table.get("partitions", []):
            source = partition.get("source", {})
            text = expression_to_text(source.get("expression"))

            if "Dataflows" in text or "dataflowId" in text:
                refs = extract_dataflow_ids_from_m(text)

                for ref in refs:
                    rows.append({
                        "Scope": "Partition",
                        "Table Name": table_name,
                        "Object Name": partition.get("name"),
                        "Source Type": source.get("type"),
                        "Connector": ref["Connector"],
                        "Workspace Id": ref["Workspace Id"],
                        "Dataflow Id": ref["Dataflow Id"],
                        "Entity": ref["Entity"],
                        "Expression Preview": text[:1500],
                    })

    for expr_obj in model.get("expressions", []):
        text = expression_to_text(expr_obj.get("expression"))

        if "Dataflows" in text or "dataflowId" in text:
            refs = extract_dataflow_ids_from_m(text)

            for ref in refs:
                rows.append({
                    "Scope": "Expression",
                    "Table Name": None,
                    "Object Name": expr_obj.get("name"),
                    "Source Type": expr_obj.get("kind"),
                    "Connector": ref["Connector"],
                    "Workspace Id": ref["Workspace Id"],
                    "Dataflow Id": ref["Dataflow Id"],
                    "Entity": ref["Entity"],
                    "Expression Preview": text[:1500],
                })

    return pd.DataFrame(rows)

def enrich_and_build_mapping(audit_df):
    workspace_cache = {}
    enriched_rows = []
    mapping_rows = []

    if audit_df.empty:
        return audit_df, pd.DataFrame()

    unique_refs = audit_df[["Workspace Id", "Dataflow Id"]].drop_duplicates()

    for _, ref in unique_refs.iterrows():
        workspace_id = ref["Workspace Id"]
        dataflow_id = ref["Dataflow Id"]

        if pd.isna(workspace_id) or pd.isna(dataflow_id):
            continue

        if workspace_id not in workspace_cache:
            workspace_cache[workspace_id] = get_all_dataflows_by_workspace(workspace_id)

        all_dataflows = workspace_cache[workspace_id]

        matched = all_dataflows[
            all_dataflows["Dataflow Id Lower"] == str(dataflow_id).lower()
        ]

        if matched.empty:
            gen1_name = None
            generation = None
            status = "NO ENCONTRADO EN WORKSPACE"
            gen2 = None
        else:
            source = matched.iloc[0].to_dict()
            gen1_name = source["Dataflow Name"]
            generation = source["Generation"]

            if str(generation).startswith("Gen1"):
                gen2 = find_gen2_candidate(gen1_name, all_dataflows)
                status = "GEN1 DETECTADO"
            else:
                gen2 = None
                status = "NO ES GEN1"

        if gen2:
            mapping_rows.append({
                "Old Dataflow Name": gen1_name,
                "Old Dataflow Id": dataflow_id,
                "Old Workspace Id": workspace_id,
                "Old Generation": generation,
                "New Dataflow Name": gen2["Dataflow Name"],
                "New Dataflow Id": gen2["Dataflow Id"],
                "New Workspace Id": workspace_id,
                "New Generation": gen2["Generation"],
                "Status": "OK",
            })
        elif status == "GEN1 DETECTADO":
            mapping_rows.append({
                "Old Dataflow Name": gen1_name,
                "Old Dataflow Id": dataflow_id,
                "Old Workspace Id": workspace_id,
                "Old Generation": generation,
                "New Dataflow Name": GEN2_NAME_MAPPING.get(gen1_name, f"{gen1_name}_1"),
                "New Dataflow Id": None,
                "New Workspace Id": workspace_id,
                "New Generation": None,
                "Status": "NO SE ENCONTRO GEN2",
            })

    mapping_df = pd.DataFrame(mapping_rows)

    for _, row in audit_df.iterrows():
        workspace_id = row["Workspace Id"]
        dataflow_id = row["Dataflow Id"]

        detected_name = None
        detected_generation = None

        if pd.notna(workspace_id) and workspace_id in workspace_cache:
            all_dataflows = workspace_cache[workspace_id]
            matched = all_dataflows[
                all_dataflows["Dataflow Id Lower"] == str(dataflow_id).lower()
            ]

            if not matched.empty:
                detected_name = matched["Dataflow Name"].iloc[0]
                detected_generation = matched["Generation"].iloc[0]

        new_row = row.to_dict()
        new_row["Detected Dataflow Name"] = detected_name
        new_row["Detected Generation"] = detected_generation
        new_row["Is Gen1"] = str(detected_generation).startswith("Gen1")
        enriched_rows.append(new_row)

    if mapping_df.empty:
        return pd.DataFrame(enriched_rows), mapping_df

    return pd.DataFrame(enriched_rows), mapping_df.drop_duplicates()

def convert_powerbi_dataflows_to_powerplatform(text):
    text = re.sub(
        r'PowerBI\.Dataflows\s*\([^)]*\)',
        'PowerPlatform.Dataflows(null)',
        text,
        flags=re.I
    )

    if 'Id="Workspaces"' in text or "Id = \"Workspaces\"" in text:
        return text

    source_match = re.search(
        r'(?m)^(\s*)(#?"?[^=\r\n]+"?)\s*=\s*PowerPlatform\.Dataflows\s*\([^)]*\)\s*,',
        text
    )

    if not source_match:
        return text

    indent = source_match.group(1)
    source_step = source_match.group(2).strip()
    source_assignment = source_match.group(0)

    workspaces_step = f'{indent}Workspaces = {source_step}{{[Id="Workspaces"]}}[Data],'
    text = text.replace(source_assignment, source_assignment + "\n" + workspaces_step, 1)

    escaped_source = re.escape(source_step)

    text = re.sub(
        rf'=\s*{escaped_source}\s*\{{\s*\[\s*workspaceId\s*=',
        '= Workspaces{[workspaceId=',
        text,
        count=1,
        flags=re.I
    )

    return text

def add_version_to_entity_if_missing(text):
    def repl(match):
        inside = match.group(1)
        if "version" in inside.lower():
            return match.group(0)
        return "[" + inside.rstrip() + ', version=""]'

    return re.sub(
        r'\[([^\]]*entity\s*=\s*"[^"]+"[^\]]*)\]',
        repl,
        text,
        flags=re.I
    )

def rewrite_text_to_gen2(text, mapping_df):
    original = text

    for _, m in mapping_df.iterrows():
        if m["Status"] != "OK":
            continue

        old_workspace_id = str(m["Old Workspace Id"])
        old_dataflow_id = str(m["Old Dataflow Id"])
        new_workspace_id = str(m["New Workspace Id"])
        new_dataflow_id = str(m["New Dataflow Id"])

        text = text.replace(old_workspace_id, new_workspace_id)
        text = text.replace(old_dataflow_id, new_dataflow_id)

    text = convert_powerbi_dataflows_to_powerplatform(text)
    text = add_version_to_entity_if_missing(text)

    return text, text != original

def update_bim_to_gen2(bim, mapping_df):
    new_bim = copy.deepcopy(bim)
    model = new_bim.get("model", {})
    changes = []

    ok_mapping = mapping_df[mapping_df["Status"] == "OK"]

    if ok_mapping.empty:
        return new_bim, pd.DataFrame()

    old_ids = set(ok_mapping["Old Dataflow Id"].astype(str).str.lower())

    for table in model.get("tables", []):
        table_name = table.get("name")

        for partition in table.get("partitions", []):
            source = partition.get("source", {})
            expr = source.get("expression")
            text = expression_to_text(expr)

            if not any(old_id in text.lower() for old_id in old_ids):
                continue

            new_text, changed = rewrite_text_to_gen2(text, mapping_df)

            if changed:
                source["expression"] = text_to_expression(expr, new_text)
                changes.append({
                    "Scope": "Partition",
                    "Table Name": table_name,
                    "Object Name": partition.get("name"),
                    "Changed": True,
                    "Before": text[:2000],
                    "After": new_text[:2000],
                })

    for expr_obj in model.get("expressions", []):
        expr = expr_obj.get("expression")
        text = expression_to_text(expr)

        if not any(old_id in text.lower() for old_id in old_ids):
            continue

        new_text, changed = rewrite_text_to_gen2(text, mapping_df)

        if changed:
            expr_obj["expression"] = text_to_expression(expr, new_text)
            changes.append({
                "Scope": "Expression",
                "Table Name": None,
                "Object Name": expr_obj.get("name"),
                "Changed": True,
                "Before": text[:2000],
                "After": new_text[:2000],
            })

    return new_bim, pd.DataFrame(changes)

workspace_name, workspace_id = resolve_workspace_name_and_id(SEMANTIC_WORKSPACE)

print("Leyendo definicion del modelo semantico...")
bim = labs.get_semantic_model_bim(
    dataset=SEMANTIC_MODEL,
    workspace=SEMANTIC_WORKSPACE
)

audit_raw = audit_model_dataflow_sources(bim)
audit, mapping = enrich_and_build_mapping(audit_raw)

show("01_origenes_dataflow_detectados_en_modelo", audit)
show("02_mapeo_gen1_a_gen2", mapping)

MIGRATION_READY = False
MIGRATION_APPLIED = False

if audit.empty:
    print("\nEl modelo no tiene referencias detectables a Dataflows en la definicion BIM/TMSL.")

elif mapping.empty:
    print("\nSe detectaron Dataflows, pero ninguno aparece como Gen1. Puede que el modelo ya este en Gen2.")

elif not mapping[mapping["Status"] != "OK"].empty:
    print("\nHay Dataflows Gen1 sin Gen2 equivalente. Revisa 02_mapeo_gen1_a_gen2.")
    display(mapping[mapping["Status"] != "OK"])

else:
    new_bim, changes = update_bim_to_gen2(bim, mapping)
    show("03_cambios_propuestos", changes)

    if changes.empty:
        print("\nNo hubo cambios. Puede que el modelo ya este en Gen2 o que el patron M sea distinto.")

    elif not APPLY_CHANGES:
        MIGRATION_READY = True
        print("\nDRY RUN: no se modifico el modelo.")
        print("Si 03_cambios_propuestos se ve bien, cambia APPLY_CHANGES = True en la Celda 1 y ejecuta de nuevo.")

    else:
        if CREATE_BACKUP_MODEL:
            print(f"\nCreando backup del modelo: {BACKUP_MODEL_NAME}")

            labs.deploy_semantic_model(
                source_dataset=SEMANTIC_MODEL,
                source_workspace=SEMANTIC_WORKSPACE,
                target_dataset=BACKUP_MODEL_NAME,
                target_workspace=SEMANTIC_WORKSPACE,
                refresh_target_dataset=False,
                overwrite=True
            )

        print("\nActualizando modelo semantico para apuntar al Dataflow Gen2...")

        labs.update_semantic_model_from_bim(
            dataset=SEMANTIC_MODEL,
            workspace=SEMANTIC_WORKSPACE,
            bim_file=new_bim
        )

        MIGRATION_READY = True
        MIGRATION_APPLIED = True

        print("\nLISTO: el modelo semantico fue actualizado para usar Dataflow Gen2.")
        print("Siguiente paso: ejecuta la Celda 3 para rebind, refresh y validacion.")


StatementMeta(, 908309ba-3c8b-4769-a868-a8529552e783, 46, Finished, Available, Finished, False)

Leyendo definicion del modelo semantico...

==================== 01_origenes_dataflow_detectados_en_modelo ====================
1 filas


,Scope,Table Name,Object Name,Source Type,Connector,Workspace Id,Dataflow Id,Entity,Expression Preview,Detected Dataflow Name,Detected Generation,Is Gen1
0,Partition,Clientes,Clientes,m,PowerPlatform.Dataflows,05a77d9e-fc1f-4127-9a04-64324d293b25,12ffbe13-c267-4aa1-b7c6-cd622b450c1d,Clientes,let\n Origen = PowerPlatform.Dataflows(null...,DatosCliente,Gen1,True



==================== 02_mapeo_gen1_a_gen2 ====================
1 filas


,Old Dataflow Name,Old Dataflow Id,Old Workspace Id,Old Generation,New Dataflow Name,New Dataflow Id,New Workspace Id,New Generation,Status
0,DatosCliente,12ffbe13-c267-4aa1-b7c6-cd622b450c1d,05a77d9e-fc1f-4127-9a04-64324d293b25,Gen1,DatosCliente_1,82d30cce-417c-4ec6-8e75-07e21790c3e8,05a77d9e-fc1f-4127-9a04-64324d293b25,Gen2 CI/CD,OK



==================== 03_cambios_propuestos ====================
1 filas


,Scope,Table Name,Object Name,Changed,Before,After
0,Partition,Clientes,Clientes,True,let\n Origen = PowerPlatform.Dataflows(null...,let\n Origen = PowerPlatform.Dataflows(null...



Creando backup del modelo: ChallengeSemnaticLink_BACKUP_ANTES_DATAFLOW_GEN2
🟢 The 'ChallengeSemnaticLink_BACKUP_ANTES_DATAFLOW_GEN2' semantic model has been created within the 'Challenge_SemanticLink_Labs' workspace.

Actualizando modelo semantico para apuntar al Dataflow Gen2...
🟢 The 'ChallengeSemnaticLink' semantic model has been updated within the 'Challenge_SemanticLink_Labs' workspace.

LISTO: el modelo semantico fue actualizado para usar Dataflow Gen2.
Siguiente paso: ejecuta la Celda 3 para rebind, refresh y validacion.


**Celda 3: Rebind + Refresh + Validación**

In [17]:
# ============================================================
# CELDA 3 - REBIND + REFRESH + VALIDACION
# ============================================================

def audit_bim_after_migration():
    bim_after = labs.get_semantic_model_bim(
        dataset=SEMANTIC_MODEL,
        workspace=SEMANTIC_WORKSPACE
    )

    rows = []

    for table in bim_after.get("model", {}).get("tables", []):
        for partition in table.get("partitions", []):
            source = partition.get("source", {})
            text = expression_to_text(source.get("expression"))

            if "Dataflows" in text or "dataflowId" in text:
                rows.append({
                    "Scope": "Partition",
                    "Table Name": table.get("name"),
                    "Object Name": partition.get("name"),
                    "Usa Gen1 PowerBI.Dataflows": "PowerBI.Dataflows" in text,
                    "Usa Gen2 PowerPlatform.Dataflows": "PowerPlatform.Dataflows" in text,
                    "Workspace Ids": re.findall(r'workspaceId\s*=\s*"([^"]+)"', text, flags=re.I),
                    "Dataflow Ids": re.findall(r'dataflowId\s*=\s*"([^"]+)"', text, flags=re.I),
                    "Entities": re.findall(r'entity\s*=\s*"([^"]+)"', text, flags=re.I),
                    "Expression Preview": text[:1500],
                })

    for expr_obj in bim_after.get("model", {}).get("expressions", []):
        text = expression_to_text(expr_obj.get("expression"))

        if "Dataflows" in text or "dataflowId" in text:
            rows.append({
                "Scope": "Expression",
                "Table Name": None,
                "Object Name": expr_obj.get("name"),
                "Usa Gen1 PowerBI.Dataflows": "PowerBI.Dataflows" in text,
                "Usa Gen2 PowerPlatform.Dataflows": "PowerPlatform.Dataflows" in text,
                "Workspace Ids": re.findall(r'workspaceId\s*=\s*"([^"]+)"', text, flags=re.I),
                "Dataflow Ids": re.findall(r'dataflowId\s*=\s*"([^"]+)"', text, flags=re.I),
                "Entities": re.findall(r'entity\s*=\s*"([^"]+)"', text, flags=re.I),
                "Expression Preview": text[:1500],
            })

    return pd.DataFrame(rows)

def build_rebind_plan(datasources, connections):
    dataflow_ref_mask = datasources.apply(
        lambda r:
            has_any(get_first_existing(r, ["Connection Kind", "Datasource Type"]), PREFERRED_CONNECTION_TYPE_KEYWORDS)
            or has_any(get_first_existing(r, ["Connection Path", "Connection URL", "Connection Details"]), PREFERRED_CONNECTION_TYPE_KEYWORDS),
        axis=1
    )

    dataflow_refs = datasources[dataflow_ref_mask].copy()

    if dataflow_refs.empty:
        print("No encontre una referencia claramente Dataflow. Usare todos los datasources para que puedas validar.")
        dataflow_refs = datasources.copy()

    show("04_referencias_a_rebind", dataflow_refs)

    candidate_rows = []

    for _, ds_row in dataflow_refs.iterrows():
        ds_connection_type = get_first_existing(ds_row, [
            "Connection Kind",
            "Datasource Type",
        ])

        ds_connection_path = get_first_existing(ds_row, [
            "Connection Path",
            "Connection URL",
            "Connection Server",
            "Connection Database",
        ])

        candidates = connections.copy()

        if PREFERRED_CONNECTION_ID:
            candidates = candidates[
                candidates["Connection Id"].astype(str).str.lower()
                == str(PREFERRED_CONNECTION_ID).lower()
            ].copy()

        if PREFERRED_CONNECTION_NAME_CONTAINS:
            candidates = candidates[
                candidates["Connection Name"].astype(str).str.contains(
                    PREFERRED_CONNECTION_NAME_CONTAINS,
                    case=False,
                    na=False
                )
            ].copy()

        if not candidates.empty:
            candidates["_Score"] = candidates.apply(
                lambda r:
                    100 * (s(r.get("Connection Type")) == ds_connection_type)
                    + 80 * (s(r.get("Connection Path")) == ds_connection_path)
                    + 50 * has_any(r.get("Connection Type"), PREFERRED_CONNECTION_TYPE_KEYWORDS)
                    + 40 * has_any(r.get("Connection Path"), PREFERRED_CONNECTION_TYPE_KEYWORDS)
                    + 20 * has_any(r.get("Connection Name"), PREFERRED_CONNECTION_TYPE_KEYWORDS)
                    + 10 * (s(r.get("Credential Type")).lower() == "oauth2")
                    + 5 * (s(r.get("Connectivity Type")) in ["ShareableCloud", "PersonalCloud", "Automatic"]),
                axis=1
            )

            candidates = candidates[candidates["_Score"] > 0].copy()
            candidates = candidates.sort_values("_Score", ascending=False)

        if candidates.empty:
            candidate_rows.append({
                "Datasource Type": ds_connection_type,
                "Datasource Path": ds_connection_path,
                "Selected Connection Id": None,
                "Selected Connection Name": None,
                "Selected Connectivity Type": None,
                "Selected Connection Type": None,
                "Selected Connection Path": None,
                "Selected Credential Type": None,
                "Score": None,
                "Status": "NO_CONNECTION_FOUND",
            })
        else:
            selected = candidates.iloc[0]

            candidate_rows.append({
                "Datasource Type": ds_connection_type,
                "Datasource Path": ds_connection_path,
                "Selected Connection Id": selected.get("Connection Id"),
                "Selected Connection Name": selected.get("Connection Name"),
                "Selected Connectivity Type": selected.get("Connectivity Type"),
                "Selected Connection Type": selected.get("Connection Type"),
                "Selected Connection Path": selected.get("Connection Path"),
                "Selected Credential Type": selected.get("Credential Type"),
                "Score": selected.get("_Score"),
                "Status": "READY_TO_BIND",
            })

    return pd.DataFrame(candidate_rows)

print("1. Validando BIM actual del modelo...")
validation_bim = audit_bim_after_migration()
show("01_validacion_bim_dataflows", validation_bim)

print("\n2. Tomando ownership si corresponde...")
if TAKE_OVER_MODEL:
    try:
        labs.takeover_item_ownership(
            item=SEMANTIC_MODEL,
            type="SemanticModel",
            workspace=SEMANTIC_WORKSPACE
        )
    except Exception as e:
        print("No se pudo hacer takeover. Si ya eres owner, no pasa nada.")
        print(type(e).__name__, str(e))
else:
    print("TAKE_OVER_MODEL = False. No se cambio ownership.")

print("\n3. Leyendo datasources del modelo semantico...")
datasources = labs.list_semantic_model_datasources(
    dataset=SEMANTIC_MODEL,
    workspace=SEMANTIC_WORKSPACE,
    expand_details=True
)
show("02_datasources_modelo", datasources)

if datasources.empty:
    raise ValueError("No se detectaron datasources en el modelo semantico.")

print("\n4. Leyendo conexiones actualmente asociadas al modelo...")
try:
    item_connections = labs.list_item_connections(
        item=SEMANTIC_MODEL,
        type="SemanticModel",
        workspace=SEMANTIC_WORKSPACE
    )
except Exception as e:
    item_connections = pd.DataFrame([{
        "Error Type": type(e).__name__,
        "Error Message": str(e),
    }])

show("03_item_connections_actuales", item_connections)

print("\n5. Leyendo todas las conexiones disponibles para tu usuario...")
connections = labs.list_connections()
show("04_conexiones_disponibles", connections)

print("\n6. Construyendo plan de rebind...")
bind_plan = build_rebind_plan(datasources, connections)
show("05_plan_rebind", bind_plan)

if bind_plan[bind_plan["Status"] == "NO_CONNECTION_FOUND"].shape[0] > 0:
    print("\nNo encontre conexion Fabric compatible ya creada.")
    print("Para OAuth/Dataflows normalmente Fabric necesita una conexion OAuth existente del usuario.")
    print("Crea/autentica la conexion una vez en Configuracion del modelo o ejecuta un refresh manual.")
    print("Luego vuelve a ejecutar esta celda.")
    raise ValueError("No hay conexion candidata para rebind.")

print("\n7. Ejecutando bind/rebind de conexion al modelo...")

if APPLY_REBIND:
    for _, row in bind_plan.iterrows():
        labs.bind_semantic_model_connection(
            dataset=SEMANTIC_MODEL,
            workspace=SEMANTIC_WORKSPACE,
            connection_id=row["Selected Connection Id"],
            connectivity_type=row["Selected Connectivity Type"],
            connection_type=row["Datasource Type"],
            connection_path=row["Datasource Path"],
        )

    print("\nRebind aplicado correctamente.")
else:
    print("\nDRY RUN: APPLY_REBIND = False. No se aplico ningun rebind.")
    print("Si 05_plan_rebind se ve bien, cambia APPLY_REBIND = True en la Celda 1 y ejecuta de nuevo.")

print("\n8. Validando conexiones despues del rebind...")
try:
    item_connections_after = labs.list_item_connections(
        item=SEMANTIC_MODEL,
        type="SemanticModel",
        workspace=SEMANTIC_WORKSPACE
    )
    show("06_item_connections_despues", item_connections_after)
except Exception as e:
    print("No se pudieron listar conexiones despues del rebind:")
    print(type(e).__name__, str(e))

if RUN_REFRESH and APPLY_REBIND:
    print("\n9. Ejecutando refresh para validar credenciales y origen...")
    try:
        refresh_result = labs.refresh_semantic_model(
            dataset=SEMANTIC_MODEL,
            workspace=SEMANTIC_WORKSPACE
        )
        print("Refresh lanzado/completado correctamente.")
        display(refresh_result)
    except Exception as e:
        print("Error en refresh:")
        print(type(e).__name__, str(e))

else:
    print("\nRUN_REFRESH = False o APPLY_REBIND = False. No se ejecuto refresh.")

print("\n10. Historial de refresh...")
try:
    history = labs.get_semantic_model_refresh_history(
        dataset=SEMANTIC_MODEL,
        workspace=SEMANTIC_WORKSPACE
    )
    show("07_refresh_history", history.head(10))
except Exception as e:
    print("No se pudo leer historial:")
    print(type(e).__name__, str(e))

print("\n11. Validacion final BIM...")
validation_bim_final = audit_bim_after_migration()
show("08_validacion_bim_final", validation_bim_final)

print("\nPROCESO TERMINADO.")


StatementMeta(, 908309ba-3c8b-4769-a868-a8529552e783, 48, Finished, Available, Finished, False)

1. Validando BIM actual del modelo...

==================== 01_validacion_bim_dataflows ====================
1 filas


,Scope,Table Name,Object Name,Usa Gen1 PowerBI.Dataflows,Usa Gen2 PowerPlatform.Dataflows,Workspace Ids,Dataflow Ids,Entities,Expression Preview
0,Partition,Clientes,Clientes,False,True,[05a77d9e-fc1f-4127-9a04-64324d293b25],[82d30cce-417c-4ec6-8e75-07e21790c3e8],[Clientes],let\n Origen = PowerPlatform.Dataflows(null...



2. Tomando ownership si corresponde...
TAKE_OVER_MODEL = False. No se cambio ownership.

3. Leyendo datasources del modelo semantico...

==================== 02_datasources_modelo ====================
1 filas


,Datasource Type,Connection Server,Connection Database,Connection Path,Connection Account,Connection Domain,Connection Kind,Connection Email Address,Connection URL,Connection Class Info,Connection Login Server,Datasource Id,Gateway Id
0,Extension,None,None,PowerPlatformDataflows,None,None,PowerPlatformDataflows,None,None,None,None,affd0103-6064-4983-8f96-a4829f053f2c,47fe2aef-b1ca-4a99-8673-a7aa1d313384


...


10. Historial de refresh...

==================== 07_refresh_history ====================
2 filas


,Request Id,Refresh Type,Start Time,End Time,Error Code,Error Description,Status,Extended Status,Attempts
0,b82250f0-f9bb-4852-94b9-a1d4d1bc0166,ViaEnhancedApi,2026-04-16T22:31:03.55Z,2026-04-16T22:31:24.133Z,None,None,Completed,Completed,"[{'attemptId': 1, 'startTime': '2026-04-16T22:..."
1,3eaaa090-8541-43ca-af9d-36aaee9051e3,ViaEnhancedApi,2026-04-16T22:15:57.697Z,2026-04-16T22:16:02.923Z,None,None,Completed,Completed,"[{'attemptId': 1, 'startTime': '2026-04-16T22:..."



11. Validacion final BIM...

==================== 08_validacion_bim_final ====================
1 filas


,Scope,Table Name,Object Name,Usa Gen1 PowerBI.Dataflows,Usa Gen2 PowerPlatform.Dataflows,Workspace Ids,Dataflow Ids,Entities,Expression Preview
0,Partition,Clientes,Clientes,False,True,[05a77d9e-fc1f-4127-9a04-64324d293b25],[82d30cce-417c-4ec6-8e75-07e21790c3e8],[Clientes],let\n Origen = PowerPlatform.Dataflows(null...



PROCESO TERMINADO.
